In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.features.strike_zone import in_strike_zone
from src.utils.temporal import split_by_date

df = load_all_snapshots()
f = add_discipline_flags(df)
swings = f[f["is_swing"]].copy()
swings["target"] = swings["is_whiff"].astype(int)
swings["in_zone_flag"] = in_strike_zone(swings)

# Normalise horizontal location to the BATTER's perspective.
# plate_x is signed from the catcher's view, so its meaning inverts
# with batter handedness — the same issue as pfx_x on Day 19.
px = pd.to_numeric(swings["plate_x"], errors="coerce").astype("float64")
swings["plate_x_bat"] = np.where(swings["stand"] == "L", -px, px)

# Height relative to the batter's own zone, so tall and short hitters
# are comparable. 0 = bottom of zone, 1 = top.
pz = pd.to_numeric(swings["plate_z"], errors="coerce").astype("float64")
top = pd.to_numeric(swings["sz_top"], errors="coerce").astype("float64")
bot = pd.to_numeric(swings["sz_bot"], errors="coerce").astype("float64")
swings["plate_z_rel"] = (pz - bot) / (top - bot)

print(swings[["plate_x_bat", "plate_z_rel"]].describe().round(3).to_string())
print()
print("plate_z_rel corr by pitch type:")
for pt, g in swings.groupby("pitch_type"):
    if len(g) < 5000:
        continue
    print(f"  {pt}: {g['plate_z_rel'].corr(g['target']):+.3f}")

       plate_x_bat  plate_z_rel
count   338223.000   338223.000
mean         0.072        0.422
std          0.595        0.401
min         -2.762       -1.754
25%         -0.337        0.149
50%          0.077        0.429
75%          0.481        0.705
max          3.074        2.213

plate_z_rel corr by pitch type:
  CH: -0.317
  CU: -0.455
  FC: -0.092
  FF: +0.214
  FS: -0.396
  KC: -0.503
  SI: +0.001
  SL: -0.382
  ST: -0.329


In [2]:
from src.utils.leakage import banned_for_pitch_outcome, check_features

split = split_by_date(swings, train_end="2024-07-14", validation_end="2024-08-31")

NUMERIC = ["plate_x_bat", "plate_z_rel", "release_speed",
           "pfx_x", "pfx_z", "release_spin_rate", "release_extension",
           "balls", "strikes"]
CATEGORICAL = ["pitch_type", "stand", "p_throws"]

def build_X(part):
    num = part[NUMERIC].apply(pd.to_numeric, errors="coerce").astype("float64")
    cat = pd.get_dummies(part[CATEGORICAL].astype(str), drop_first=True)
    X = pd.concat([num, cat], axis=1)
    return X

X_train = build_X(split.train)
X_val = build_X(split.validation).reindex(columns=X_train.columns, fill_value=0)

# LEAKAGE GUARD — the last line of defence before fitting
check_features(X_train.columns, banned_for_pitch_outcome(), context="whiff logistic")

y_train = split.train["target"].to_numpy()
y_val = split.validation["target"].to_numpy()

print(f"features: {X_train.shape[1]}")
print(f"train {X_train.shape}, val {X_val.shape}")
print(f"missing in train: {X_train.isna().sum().sum():,}")

features: 26
train (200735, 26), val (83492, 26)
missing in train: 2,423


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

def fit_eval(X_tr, y_tr, X_va, y_va, name, seed=42):
    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
    ])
    pipe.fit(X_tr, y_tr)
    p = pipe.predict_proba(X_va)[:, 1]
    return pipe, {
        "model": name,
        "log_loss": log_loss(y_va, p),
        "brier": brier_score_loss(y_va, p),
        "auc": roc_auc_score(y_va, p),
    }

model_a, res_a = fit_eval(X_train, y_train, X_val, y_val, "logistic A (linear)")
print(pd.DataFrame([res_a]).round(5).to_string(index=False))
print()
print("baseline to beat: log_loss 0.48524, brier 0.15629, auc 0.70892")

              model  log_loss  brier     auc
logistic A (linear)   0.51151 0.1644 0.65086

baseline to beat: log_loss 0.48524, brier 0.15629, auc 0.70892


In [4]:
def add_interactions(X, part):
    """pitch_type x plate_z_rel, because the sign of the height effect
    inverts between fastballs and breaking balls (Day 25)."""
    out = X.copy()
    pz = pd.to_numeric(part["plate_z_rel"], errors="coerce").astype("float64")
    for col in [c for c in X.columns if c.startswith("pitch_type_")]:
        out[f"{col}_x_pz"] = X[col].astype(float) * pz.to_numpy()
    return out

X_train_i = add_interactions(X_train, split.train)
X_val_i = add_interactions(X_val, split.validation).reindex(
    columns=X_train_i.columns, fill_value=0)

model_c, res_c = fit_eval(X_train_i, y_train, X_val_i, y_val, "logistic C (interactions)")

comparison = pd.DataFrame([
    {"model": "constant", "log_loss": 0.54072, "brier": 0.17775, "auc": np.nan},
    {"model": "lookup+zone (baseline)", "log_loss": 0.48524, "brier": 0.15629, "auc": 0.70892},
    res_a,
    res_c,
])
print(comparison.round(5).to_string(index=False))

                    model  log_loss   brier     auc
                 constant   0.54072 0.17775     NaN
   lookup+zone (baseline)   0.48524 0.15629 0.70892
      logistic A (linear)   0.51151 0.16440 0.65086
logistic C (interactions)   0.47650 0.15088 0.72245


In [5]:
coef = pd.Series(
    model_c.named_steps["clf"].coef_[0],
    index=X_train_i.columns,
).sort_values(key=abs, ascending=False)

print("=== largest coefficients (standardised features) ===")
print(coef.head(15).round(3).to_string())
print()
print("=== plate_z interactions ===")
print(coef[coef.index.str.contains("pz")].round(3).to_string())

=== largest coefficients (standardised features) ===
pitch_type_FF_x_pz    1.647
pitch_type_FF        -1.305
plate_z_rel          -1.044
pitch_type_SI        -0.626
pitch_type_SI_x_pz    0.546
pitch_type_FC_x_pz    0.343
plate_x_bat           0.222
pitch_type_FC        -0.199
pfx_z                 0.193
strikes              -0.190
release_speed         0.132
release_spin_rate     0.094
pitch_type_SL         0.085
pitch_type_CU_x_pz   -0.077
pitch_type_SC_x_pz   -0.076

=== plate_z interactions ===
pitch_type_FF_x_pz    1.647
pitch_type_SI_x_pz    0.546
pitch_type_FC_x_pz    0.343
pitch_type_CU_x_pz   -0.077
pitch_type_SC_x_pz   -0.076
pitch_type_KC_x_pz   -0.066
pitch_type_FA_x_pz    0.060
pitch_type_EP_x_pz   -0.053
pitch_type_FS_x_pz   -0.047
pitch_type_ST_x_pz    0.024
pitch_type_SV_x_pz   -0.024
pitch_type_CS_x_pz   -0.023
pitch_type_KN_x_pz    0.011
pitch_type_SL_x_pz   -0.007
pitch_type_FO_x_pz    0.006


In [6]:
coef = pd.Series(
    model_c.named_steps["clf"].coef_[0],
    index=X_train_i.columns,
).sort_values(key=abs, ascending=False)

print("=== largest coefficients ===")
print(coef.head(15).round(3).to_string())
print()
print("=== plate_z interactions (sign should flip FF vs breaking balls) ===")
print(coef[coef.index.str.endswith("_x_pz")].sort_values().round(3).to_string())
print()
print("main effect plate_z_rel:", round(coef["plate_z_rel"], 3))

=== largest coefficients ===
pitch_type_FF_x_pz    1.647
pitch_type_FF        -1.305
plate_z_rel          -1.044
pitch_type_SI        -0.626
pitch_type_SI_x_pz    0.546
pitch_type_FC_x_pz    0.343
plate_x_bat           0.222
pitch_type_FC        -0.199
pfx_z                 0.193
strikes              -0.190
release_speed         0.132
release_spin_rate     0.094
pitch_type_SL         0.085
pitch_type_CU_x_pz   -0.077
pitch_type_SC_x_pz   -0.076

=== plate_z interactions (sign should flip FF vs breaking balls) ===
pitch_type_CU_x_pz   -0.077
pitch_type_SC_x_pz   -0.076
pitch_type_KC_x_pz   -0.066
pitch_type_EP_x_pz   -0.053
pitch_type_FS_x_pz   -0.047
pitch_type_SV_x_pz   -0.024
pitch_type_CS_x_pz   -0.023
pitch_type_SL_x_pz   -0.007
pitch_type_FO_x_pz    0.006
pitch_type_KN_x_pz    0.011
pitch_type_ST_x_pz    0.024
pitch_type_FA_x_pz    0.060
pitch_type_FC_x_pz    0.343
pitch_type_SI_x_pz    0.546
pitch_type_FF_x_pz    1.647

main effect plate_z_rel: -1.044


In [7]:
X_train_raw = X_train.copy()
X_val_raw = X_val.copy()
X_train_raw["plate_x_bat"] = pd.to_numeric(split.train["plate_x"], errors="coerce")
X_val_raw["plate_x_bat"] = pd.to_numeric(split.validation["plate_x"], errors="coerce")

_, res_no_norm = fit_eval(X_train_raw, y_train, X_val_raw, y_val, "A without plate_x normalisation")
print(pd.DataFrame([res_no_norm, res_a]).round(5).to_string(index=False))

                          model  log_loss   brier     auc
A without plate_x normalisation   0.51272 0.16471 0.64721
            logistic A (linear)   0.51151 0.16440 0.65086


In [8]:
print(split.train["pitch_type"].value_counts())

pitch_type
FF    64478
SI    30800
SL    30315
CH    21514
FC    17193
ST    14257
CU    10705
FS     6788
KC     3052
SV      847
KN      312
FA      160
EP      107
FO       66
CS       14
SC        6
Name: count, dtype: int64
